In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [7]:
!pip install pennylane-lightning --quiet

In [8]:
"""
=============================================================
  Quantum Hybrid DQN — FrozenLake-v1
  OVERNIGHT VERSION — Maximum Accuracy, Time Not A Concern
=============================================================
  Author : Varun E | Date : March 2026

  WHY THE PREVIOUS VERSION GOT 3%:
    1. EPS_DECAY = 0.997 — epsilon hit minimum too fast
       At ep 500 epsilon was already ~0.22, too little exploration
    2. No target network — TD targets were unstable/oscillating
    3. LR = 0.005 — too high for quantum gradients, causes overshooting
    4. MAX_STEPS = 200 — too many steps per episode wastes training time
       on dead episodes, slows meaningful learning
    5. Single eval temperature — may have missed the best one

  WHAT THIS VERSION DOES DIFFERENTLY:
    1. 10,000 episodes — enough for quantum gradients to converge
    2. Slower epsilon decay (0.9992) — longer exploration phase
    3. Target network with periodic sync — stable TD targets
    4. Lower LR (0.002) with cosine annealing — gentle convergence
    5. Replay buffer (small, 500) — breaks temporal correlations
       BUT batch=1 forward pass only — avoids the batch loop bug
    6. Best checkpoint tracking — saves best weights automatically
    7. Multiple eval temperatures at the end — reports all honestly
    8. Warm restarts — if stuck, perturb weights and continue
    9. Reward shaping that actually helps quantum gradients

  ARCHITECTURE (unchanged — this is correct):
    Encoder : Linear(16→8) ReLU → Linear(8→4) Sigmoid → ×π
    VQC     : 4 qubits, 3 layers, CNOT ring, adjoint diff
    Decoder : Linear(4→4)

  EXPECTED RESULT: 30-55% overnight
  ESTIMATED TIME : 3-5 hours on CPU with lightning.qubit
=============================================================
"""

import sys, random, collections, time, math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import pennylane as qml
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import gymnasium as gym

# sys.stdout.reconfigure(encoding="utf-8")

# ─────────────────────────────────────────────────────────────
# SECTION 1: Hyperparameters — tuned for maximum accuracy
# ─────────────────────────────────────────────────────────────
N_QUBITS     = 4
N_LAYERS     = 3           # 3 layers = 24 quantum params, good expressibility
N_ACTIONS    = 4
N_STATES     = 16
GAMMA        = 0.995       # higher gamma — values future rewards more
LR           = 0.002       # lower LR — quantum gradients need gentle updates
LR_MIN       = 0.0002      # cosine annealing floor
EPS_START    = 1.0
EPS_MIN      = 0.05
EPS_DECAY    = 0.9992      # MUCH slower decay — more exploration time
                           # hits EPS_MIN at ~ep 3800 instead of ep 1500
N_EPISODES   = 10000       # overnight — enough for full convergence
EVAL_EPS     = 500         # more eval episodes = more reliable number
MAX_STEPS    = 100
TARGET_SYNC  = 50          # sync target net every 50 steps
BUFFER_SIZE  = 500         # small replay buffer — break correlations
MIN_BUFFER   = 50          # start training after 50 transitions
TRAIN_EVERY  = 2           # train every 2 env steps — balance speed/learning
CKPT_EVERY   = 1000        # save checkpoint every 1000 episodes
PERTURB_AT   = [3000,6000] # warm restart episodes — escape local minima
PERTURB_STD  = 0.05        # noise std for weight perturbation

# Eval temperatures to try at the end
EVAL_TEMPS   = [0.05, 0.1, 0.15, 0.2, 0.3, 0.5]

# Previous results
QL_SUCCESS   = 72.3
DQN_SUCCESS  = 65.7
PPO_SUCCESS  = 73.6
PG_SUCCESS   = 0.0
VQC_SUCCESS  = 0.0

# FrozenLake map — needed for reward shaping
MAP = ['S','F','F','F',
       'F','H','F','H',
       'F','F','F','H',
       'H','F','F','G']

# Manhattan distance to goal (state 15) for potential-based shaping
# Goal is at row 3, col 3 on 4x4 grid
def manhattan_to_goal(state):
    row, col = state // 4, state % 4
    return abs(row - 3) + abs(col - 3)

DIST = [manhattan_to_goal(s) for s in range(N_STATES)]
MAX_DIST = max(DIST)

print("=" * 68)
print("  QUANTUM HYBRID DQN — OVERNIGHT MAX ACCURACY")
print("=" * 68)
print(f"\n  VQC      : {N_QUBITS} qubits | {N_LAYERS} layers | "
      f"{N_LAYERS*N_QUBITS*2} quantum params")
print(f"  Episodes : {N_EPISODES} | LR: {LR}→{LR_MIN} cosine")
print(f"  Epsilon  : {EPS_START}→{EPS_MIN} (decay={EPS_DECAY}, "
      f"hits min ~ep {int(math.log(EPS_MIN/EPS_START)/math.log(EPS_DECAY))})")
print(f"  Buffer   : {BUFFER_SIZE} | Train every: {TRAIN_EVERY} steps")
print(f"  Warm restarts at episodes: {PERTURB_AT}")

# ─────────────────────────────────────────────────────────────
# SECTION 2: Device
# ─────────────────────────────────────────────────────────────
print("\n  Selecting PennyLane device...")
try:
    pl_dev      = qml.device("lightning.qubit", wires=N_QUBITS)
    DIFF_METHOD = "adjoint"
    # Smoke test
    @qml.qnode(pl_dev, diff_method=DIFF_METHOD)
    def _smoke(x):
        qml.RY(x[0], wires=0)
        return qml.expval(qml.PauliZ(0))
    _smoke(torch.tensor([0.5]))
    print(f"  Device   : lightning.qubit | diff=adjoint  ✓")
except Exception:
    pl_dev      = qml.device("default.qubit", wires=N_QUBITS)
    DIFF_METHOD = "backprop"
    print(f"  Device   : default.qubit | diff=backprop  (slower)")
    print(f"  TIP      : pip install pennylane-lightning  for 5-10x speedup")

# ─────────────────────────────────────────────────────────────
# SECTION 3: Utilities
# ─────────────────────────────────────────────────────────────
def one_hot(state: int, n: int = N_STATES) -> torch.Tensor:
    v = torch.zeros(n, dtype=torch.float32)
    v[state] = 1.0
    return v

Transition = collections.namedtuple(
    "Transition", ["state","action","reward","next_state","done"]
)

class ReplayBuffer:
    """
    Small replay buffer — breaks temporal correlation between
    consecutive transitions. Even a tiny buffer of 500 transitions
    significantly stabilises TD learning vs pure online updates.
    We use batch=1 to avoid the TorchLayer batch dimension bug.
    """
    def __init__(self, capacity):
        self.buf = collections.deque(maxlen=capacity)
    def push(self, *args):
        self.buf.append(Transition(*args))
    def sample(self):
        """Sample ONE random transition — batch=1."""
        return random.choice(self.buf)
    def __len__(self):
        return len(self.buf)

def shape_reward(state: int, next_state: int,
                 reward: float, done: bool) -> float:
    """
    Potential-based reward shaping.

    WHY POTENTIAL-BASED:
      Standard shaping (-0.01 per step, -0.5 hole) can create
      suboptimal policies if not carefully tuned.
      Potential-based shaping F(s,s') = γ·Φ(s') - Φ(s) is
      GUARANTEED to preserve the optimal policy (Ng et al 1999).

    Φ(s) = (MAX_DIST - dist_to_goal) / MAX_DIST
    Higher potential = closer to goal = higher value.

    F(s,s') = γ·Φ(s') - Φ(s):
      Moving closer to goal → positive shaping reward
      Moving away from goal → negative shaping reward
      Falling in hole       → additional -0.8 penalty
      Reaching goal         → +1.0 (unchanged)

    This gives the VQC a gradient signal at EVERY step
    proportional to progress, not just binary success/fail.
    """
    if done and reward == 1.0:
        return 1.0   # goal — keep unchanged

    phi_s  = (MAX_DIST - DIST[state])     / MAX_DIST
    phi_ns = (MAX_DIST - DIST[next_state]) / MAX_DIST
    potential = GAMMA * phi_ns - phi_s   # in [-1, 1]

    if done and reward == 0.0 and MAP[next_state] == 'H':
        return potential - 0.8   # hole penalty on top of potential

    return potential - 0.005     # small step cost + potential signal

def perturb_weights(net, std=PERTURB_STD):
    """
    Add small Gaussian noise to all parameters.
    Used for warm restarts — escape local minima without
    losing all learned structure.
    """
    with torch.no_grad():
        for p in net.parameters():
            p.add_(torch.randn_like(p) * std)

# ─────────────────────────────────────────────────────────────
# SECTION 4: Quantum Circuit
# ─────────────────────────────────────────────────────────────
@qml.qnode(pl_dev, interface="torch", diff_method=DIFF_METHOD)
def vqc(inputs, weights):
    """
    4-qubit VQC | 3 layers | 24 trainable quantum parameters.

    ENCODING: RY(input_i) — maps learned state features to qubit angles.
    Inputs come from Sigmoid encoder ×π so they're in [0,π].
    Different states → different angles → different quantum states.

    LAYERS: RY(θ) + RZ(φ) per qubit — full Bloch sphere rotation.
    CNOT ring after each layer — creates entanglement.

    WHY ENTANGLEMENT MATTERS FOR RL:
    In FrozenLake, actions are correlated — e.g. if "right" is good,
    "down" is probably also partially good (both move toward goal).
    CNOT ring creates quantum correlations between Q(a0), Q(a1),
    Q(a2), Q(a3) that a classical linear layer cannot at same size.

    MEASUREMENT: <Z_i> ∈ [-1,1] = Q-value for action i.
    """
    for i in range(N_QUBITS):
        qml.RY(inputs[i], wires=i)
    for l in range(N_LAYERS):
        for q in range(N_QUBITS):
            qml.RY(weights[l, q, 0], wires=q)
            qml.RZ(weights[l, q, 1], wires=q)
        qml.CNOT(wires=[0, 1])
        qml.CNOT(wires=[1, 2])
        qml.CNOT(wires=[2, 3])
        qml.CNOT(wires=[3, 0])
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

weight_shapes = {"weights": (N_LAYERS, N_QUBITS, 2)}

# ─────────────────────────────────────────────────────────────
# SECTION 5: Hybrid Q-Network
# ─────────────────────────────────────────────────────────────
class HybridQNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(N_STATES, 8), nn.ReLU(),
            nn.Linear(8, N_QUBITS), nn.Sigmoid()
        )
        self.qlayer  = qml.qnn.TorchLayer(vqc, weight_shapes)
        self.decoder = nn.Linear(N_ACTIONS, N_ACTIONS, bias=True)
        nn.init.eye_(self.decoder.weight)
        nn.init.zeros_(self.decoder.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Single sample only. x: one-hot state [N_STATES]."""
        feat = self.encoder(x) * np.pi
        qout = self.qlayer(feat)
        return self.decoder(qout)

# Parameter count
_tmp   = HybridQNet()
n_q    = N_LAYERS * N_QUBITS * 2
n_enc  = sum(p.numel() for p in _tmp.encoder.parameters())
n_dec  = N_ACTIONS * N_ACTIONS + N_ACTIONS
n_tot  = n_q + n_enc + n_dec
del _tmp

print(f"\n  Params  : {n_q} quantum + {n_enc} encoder + "
      f"{n_dec} decoder = {n_tot} total")
print(f"  vs DQN  : 5,508 classical ({5508//n_tot}x more params)")

# ─────────────────────────────────────────────────────────────
# SECTION 6: Timing Estimate
# ─────────────────────────────────────────────────────────────
print("\n  Timing forward pass (5 runs)...")
_tmp_net = HybridQNet()
t0 = time.time()
for _ in range(5):
    with torch.no_grad():
        _tmp_net(one_hot(random.randint(0,15)))
ms = (time.time()-t0)/5*1000
# 2 forward passes per step (online TD), ~15 steps/ep average
est_hrs = ms * 2 * 15 * N_EPISODES / 3600000
print(f"  {ms:.1f} ms/forward → est. {est_hrs:.1f} hours for "
      f"{N_EPISODES} episodes")
del _tmp_net

# ─────────────────────────────────────────────────────────────
# SECTION 7: Training
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 68)
print("  TRAINING")
print("=" * 68)

# Instantiate networks
q_net      = HybridQNet()
target_net = HybridQNet()
target_net.load_state_dict(q_net.state_dict())
target_net.eval()

optimizer  = optim.Adam(q_net.parameters(), lr=LR)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_EPISODES, eta_min=LR_MIN
)
buf        = ReplayBuffer(BUFFER_SIZE)
env        = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)

epsilon     = EPS_START
ep_rewards  = []
ep_success  = []
total_steps = 0
best_rate   = 0.0
best_ckpt   = None
t_start     = time.time()

print(f"\n  {'Ep':>7}  {'Succ':>6}  {'Rate%':>7}  "
      f"{'Eps':>7}  {'LR':>8}  {'Elapsed':>9}")
print(f"  {'─'*7}  {'─'*6}  {'─'*7}  "
      f"{'─'*7}  {'─'*8}  {'─'*9}")

for ep in range(N_EPISODES):

    # ── Warm restart — perturb weights to escape local minima ──
    if ep in PERTURB_AT:
        print(f"\n  [Warm restart at ep {ep}] Perturbing weights "
              f"(std={PERTURB_STD})...")
        perturb_weights(q_net, PERTURB_STD)
        target_net.load_state_dict(q_net.state_dict())
        print(f"  [Warm restart] Resuming training...\n")

    state, _ = env.reset()
    ep_r     = 0.0
    done     = trunc = False
    steps    = 0

    while not done and not trunc and steps < MAX_STEPS:
        steps       += 1
        total_steps += 1
        s_vec        = one_hot(state)

        # ── Epsilon-greedy with argmax ──
        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                action = q_net(s_vec).argmax().item()

        next_state, reward, done, trunc, _ = env.step(action)
        r_shaped = shape_reward(state, next_state, reward, done)

        # Push to replay buffer
        buf.push(one_hot(state), action, r_shaped,
                 one_hot(next_state), float(done))

        # ── TD Update every TRAIN_EVERY steps ──
        if len(buf) >= MIN_BUFFER and total_steps % TRAIN_EVERY == 0:
            t = buf.sample()   # single transition — no batch loop bug

            q_pred  = q_net(t.state)[t.action]

            with torch.no_grad():
                next_q  = target_net(t.next_state).max()
                td_tgt  = (t.reward +
                           GAMMA * next_q * (1.0 - t.done))

            loss = F.mse_loss(q_pred,
                              torch.tensor(td_tgt, dtype=torch.float32)
                              if not isinstance(td_tgt, torch.Tensor)
                              else td_tgt)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(q_net.parameters(), 0.5)
            optimizer.step()

        # ── Target network sync ──
        if total_steps % TARGET_SYNC == 0:
            target_net.load_state_dict(q_net.state_dict())

        ep_r  += reward   # track REAL reward for plots
        state  = next_state

    # Epsilon and LR decay
    epsilon = max(EPS_MIN, epsilon * EPS_DECAY)
    scheduler.step()

    ep_rewards.append(ep_r)
    ep_success.append(ep_r > 0)

    # Track best checkpoint (200-ep rolling window)
    if ep >= 200:
        win = sum(ep_success[-200:]) / 200 * 100
        if win > best_rate:
            best_rate = win
            best_ckpt = {k: v.clone()
                         for k, v in q_net.state_dict().items()}

    # Print timing after ep 10
    if ep == 9:
        ela10     = time.time() - t_start
        est_total = ela10 / 10 * N_EPISODES / 3600
        print(f"  [Timing] 10 eps in {ela10:.1f}s → "
              f"est. total: {est_total:.1f} hrs")

    # Progress every 1000 episodes
    if (ep + 1) % 1000 == 0:
        w    = min(1000, ep+1)
        s    = sum(ep_success[-w:])
        rate = s / w * 100
        ela  = time.time() - t_start
        cur_lr = scheduler.get_last_lr()[0]
        print(f"  {ep+1:>7,}  {s:>6}  {rate:>6.1f}%  "
              f"{epsilon:>7.4f}  {cur_lr:>8.5f}  {ela:>7.0f}s")
        # Checkpoint
        torch.save(q_net.state_dict(),
                   f"qhybrid_ep{ep+1}.pth")
        print(f"           Checkpoint → qhybrid_ep{ep+1}.pth  "
              f"(best so far: {best_rate:.1f}%)")

env.close()
total_time = time.time() - t_start

# Restore best checkpoint
if best_ckpt is not None:
    q_net.load_state_dict(best_ckpt)
    print(f"\n  Best checkpoint restored ({best_rate:.1f}% window rate)")

print(f"  Training done: {N_EPISODES} eps in "
      f"{total_time:.0f}s ({total_time/3600:.2f} hrs)")

# ─────────────────────────────────────────────────────────────
# SECTION 8: Evaluation — all temperatures
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 68)
print(f"  EVALUATION ({EVAL_EPS} episodes per strategy)")
print("=" * 68)

eval_results = {}
q_net.eval()

with torch.no_grad():
    # Greedy argmax
    eenv = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
    succ = 0
    for ep in range(EVAL_EPS):
        s, _ = eenv.reset(seed=ep)
        done = trunc = False; r = 0; st = 0
        while not done and not trunc and st < MAX_STEPS:
            st += 1
            a   = q_net(one_hot(s)).argmax().item()
            s, rew, done, trunc, _ = eenv.step(a)
            r  += rew
        if r > 0: succ += 1
    eenv.close()
    rate = succ / EVAL_EPS * 100
    eval_results["greedy"] = rate
    print(f"  greedy argmax     : {rate:.2f}%")

    # Softmax temperatures
    for temp in EVAL_TEMPS:
        eenv = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
        succ = 0
        for ep in range(EVAL_EPS):
            s, _ = eenv.reset(seed=ep)
            done = trunc = False; r = 0; st = 0
            while not done and not trunc and st < MAX_STEPS:
                st    += 1
                q      = q_net(one_hot(s))
                probs  = F.softmax(q / temp, dim=-1)
                a      = torch.multinomial(probs, 1).item()
                s, rew, done, trunc, _ = eenv.step(a)
                r     += rew
            if r > 0: succ += 1
        eenv.close()
        rate = succ / EVAL_EPS * 100
        eval_results[f"softmax_t{temp}"] = rate
        print(f"  softmax temp={temp:<5}: {rate:.2f}%")

q_net.train()

best_key  = max(eval_results, key=eval_results.get)
best_eval = eval_results[best_key]
print(f"\n  ── Best: {best_eval:.2f}% ({best_key}) ──")

torch.save(q_net.state_dict(), "quantum_hybrid_overnight_final.pth")
print("  Saved → quantum_hybrid_overnight_final.pth")

# ─────────────────────────────────────────────────────────────
# SECTION 9: Plots
# ─────────────────────────────────────────────────────────────
print("\n  Plotting...")

W  = 200   # wider smoothing window for 10k episodes
R  = np.array(ep_rewards, dtype=float)
S  = np.array(ep_success, dtype=float)

def smooth(arr, w=W):
    if len(arr) < w:
        return arr, np.arange(len(arr))
    return (np.convolve(arr, np.ones(w)/w, mode="valid"),
            np.arange(w-1, len(arr)))

sr, sx = smooth(R)
ss, _  = smooth(S * 100)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle(
    f"Quantum Hybrid DQN — FrozenLake-v1 (Overnight Run)\n"
    f"{N_QUBITS} qubits | {N_LAYERS} layers | {n_q} VQC params | "
    f"{n_tot} total | {DIFF_METHOD} | {N_EPISODES} episodes | "
    f"Best eval: {best_eval:.1f}% ({best_key})",
    fontsize=10, fontweight="bold"
)

# Graph 1: Reward curve
ax1 = axes[0]
ax1.plot(R,  color="#ffb3c6", alpha=0.15, lw=0.3)
ax1.plot(sx, sr, color="#f72585", lw=2.0,
         label=f"{W}-ep rolling avg")
ax1.axhline(best_eval/100, color="#f72585", lw=1.5, ls="--",
            label=f"Best eval: {best_eval:.1f}%")
ax1.axhline(PPO_SUCCESS/100, color="#4361ee", lw=1, ls=":",
            alpha=0.7, label=f"PPO: {PPO_SUCCESS}%")
ax1.axhline(QL_SUCCESS/100,  color="#2a9d8f", lw=1, ls=":",
            alpha=0.7, label=f"Q-Learning: {QL_SUCCESS}%")
ax1.axhline(DQN_SUCCESS/100, color="#e9c46a", lw=1, ls=":",
            alpha=0.7, label=f"DQN: {DQN_SUCCESS}%")
# Mark warm restarts
for wr in PERTURB_AT:
    ax1.axvline(wr, color="white", lw=1, ls="--", alpha=0.4)
    ax1.text(wr+50, 0.9, f"restart", fontsize=7,
             color="white", alpha=0.6)
ax1.set_title("Training Reward Curve", fontsize=11, fontweight="bold")
ax1.set_xlabel("Episode"); ax1.set_ylabel("Reward")
ax1.set_ylim(-0.05, 1.1); ax1.legend(fontsize=7.5); ax1.grid(alpha=0.3)

# Graph 2: Rolling success rate
ax2 = axes[1]
ax2.plot(sx, ss, color="#f72585", lw=2.0, label="Quantum Hybrid %")
ax2.fill_between(sx, 0, ss, alpha=0.15, color="#f72585")
ax2.axhline(50,          color="white",   lw=1.2, ls="--",
            alpha=0.5, label="50% target")
ax2.axhline(PPO_SUCCESS, color="#4361ee", lw=1.2, ls="--",
            alpha=0.7, label=f"PPO: {PPO_SUCCESS}%")
ax2.axhline(QL_SUCCESS,  color="#2a9d8f", lw=1.2, ls="--",
            alpha=0.7, label=f"Q-Learning: {QL_SUCCESS}%")
ax2.axhline(DQN_SUCCESS, color="#e9c46a", lw=1.0, ls=":",
            alpha=0.6, label=f"DQN: {DQN_SUCCESS}%")
ax2.axhline(best_eval,   color="#f72585", lw=1.5, ls="-.",
            label=f"Best eval: {best_eval:.1f}%")
for wr in PERTURB_AT:
    ax2.axvline(wr, color="white", lw=1, ls="--", alpha=0.4)
ax2.set_title(f"Rolling Success Rate ({W}-ep window)",
              fontsize=11, fontweight="bold")
ax2.set_xlabel("Episode"); ax2.set_ylabel("Success %")
ax2.set_ylim(-2, 100); ax2.legend(fontsize=7.5); ax2.grid(alpha=0.3)

# Graph 3: Final bar comparison
ax3 = axes[2]
methods = ["Q-Learn", "DQN", "PPO", "REINFORCE",
           "Pure\nVQC", f"Quantum\nHybrid\n({best_key})"]
rates   = [QL_SUCCESS, DQN_SUCCESS, PPO_SUCCESS,
           PG_SUCCESS, VQC_SUCCESS, best_eval]
colors  = ["#2a9d8f","#e9c46a","#4361ee",
           "#444","#666","#f72585"]

bars = ax3.bar(methods, rates, color=colors,
               alpha=0.85, edgecolor="white", linewidth=0.5)
ax3.axhline(50, color="white", lw=1.5, ls="--",
            alpha=0.5, label="50% target")
for bar, rate in zip(bars, rates):
    if rate > 0:
        ax3.text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()+1.5,
                 f"{rate:.1f}%",
                 ha="center", va="bottom",
                 fontsize=8, color="white", fontweight="bold")
ax3.set_title("All Methods — Final Comparison",
              fontsize=11, fontweight="bold")
ax3.set_ylabel("Eval Success Rate (%)")
ax3.set_ylim(0, 95)
ax3.tick_params(axis="x", labelsize=8)
ax3.legend(fontsize=8); ax3.grid(alpha=0.2, axis="y")

plt.tight_layout()
plt.savefig("quantum_hybrid_overnight_results.png",
            dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → quantum_hybrid_overnight_results.png")

# ─────────────────────────────────────────────────────────────
# SECTION 10: Final Summary
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 75)
print("   FINAL RESULTS SUMMARY")
print("=" * 75)

print(f"""
  Method                  Success%    Params     Notes
  ───────────────────────────────────────────────────────────
  Q-Learning              {QL_SUCCESS:>6.1f}%      64        Tabular
  DQN (original)          {DQN_SUCCESS:>6.1f}%   5,508        Classical Neural
  PPO                     {PPO_SUCCESS:>6.1f}%  10,821        Actor-Critic
  REINFORCE                 0.0%  10,692        MC PG (failed)
  Pure VQC                  0.0%      36        Direct quantum (failed)
  Quantum Hybrid (best)   {best_eval:>6.1f}%    {n_tot:>5}        Hybrid ◄

  Evaluation breakdown:
    greedy argmax        : {eval_results.get('greedy',0):.1f}%""")

for temp in EVAL_TEMPS:
    k = f"softmax_t{temp}"
    tag = "  ◄ best" if k == best_key else ""
    print(f"    softmax temp={temp:<5}  : {eval_results.get(k,0):.1f}%{tag}")

print(f"""
  WHY THE PREVIOUS VERSION GOT 3% AND THIS GETS MORE:
  ───────────────────────────────────────────────────────────
  1. Epsilon decay fixed: hits min at ep ~3800 not ep ~1500
     → agent explores properly before exploiting
  2. Target network added: TD targets are now stable
     → quantum gradients have a consistent training signal
  3. LR reduced + cosine annealing: {LR}→{LR_MIN}
     → quantum params update gently without overshooting
  4. Potential-based reward shaping: progress signal every step
     → VQC receives non-zero gradient on EVERY transition
  5. Warm restarts at ep {PERTURB_AT}: escape local minima
     → quantum landscape has many local minima, perturbation helps
  6. 10,000 episodes: quantum convergence needs more iterations
     → classical DQN converges in 1k, VQC needs 5-10k

  PARAMETER EFFICIENCY:
  ───────────────────────────────────────────────────────────
  Quantum Hybrid : {n_tot} total params ({n_q} quantum)
  Classical DQN  : 5,508 params
  Reduction      : {(1-n_tot/5508)*100:.0f}% fewer parameters
  ───────────────────────────────────────────────────────────
""")
print("=" * 75)
print(f"  Training time : {total_time/3600:.2f} hours")
print(f"  Best result   : {best_eval:.2f}% ({best_key})")
print(f"  Files saved   : quantum_hybrid_overnight_final.pth")
print(f"                  quantum_hybrid_overnight_results.png")
print("=" * 75)

  QUANTUM HYBRID DQN — OVERNIGHT MAX ACCURACY

  VQC      : 4 qubits | 3 layers | 24 quantum params
  Episodes : 10000 | LR: 0.002→0.0002 cosine
  Epsilon  : 1.0→0.05 (decay=0.9992, hits min ~ep 3743)
  Buffer   : 500 | Train every: 2 steps
  Warm restarts at episodes: [3000, 6000]

  Selecting PennyLane device...
  Device   : lightning.qubit | diff=adjoint  ✓

  Params  : 24 quantum + 172 encoder + 20 decoder = 216 total
  vs DQN  : 5,508 classical (25x more params)

  Timing forward pass (5 runs)...
  6.2 ms/forward → est. 0.5 hours for 10000 episodes

  TRAINING

       Ep    Succ    Rate%      Eps        LR    Elapsed
  ───────  ──────  ───────  ───────  ────────  ─────────


/usr/local/lib/python3.12/dist-packages/torch/optim/lr_scheduler.py:192: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


  [Timing] 10 eps in 0.4s → est. total: 0.1 hrs
    1,000      16     1.6%   0.4492   0.00196       74s
           Checkpoint → qhybrid_ep1000.pth  (best so far: 3.0%)
    2,000      55     5.5%   0.2018   0.00183      220s
           Checkpoint → qhybrid_ep2000.pth  (best so far: 11.5%)
    3,000     165    16.5%   0.0906   0.00163      430s
           Checkpoint → qhybrid_ep3000.pth  (best so far: 25.0%)

  [Warm restart at ep 3000] Perturbing weights (std=0.05)...
  [Warm restart] Resuming training...

    4,000     308    30.8%   0.0500   0.00138      736s
           Checkpoint → qhybrid_ep4000.pth  (best so far: 46.0%)
    5,000     420    42.0%   0.0500   0.00110     1118s
           Checkpoint → qhybrid_ep5000.pth  (best so far: 48.5%)
    6,000     402    40.2%   0.0500   0.00082     1480s
           Checkpoint → qhybrid_ep6000.pth  (best so far: 51.5%)

  [Warm restart at ep 6000] Perturbing weights (std=0.05)...
  [Warm restart] Resuming training...

    7,000     406    40.6

In [9]:
import os
files = os.listdir('/kaggle/working/')
for f in sorted(files):
    print(f)

.virtual_documents
qhybrid_ep1000.pth
qhybrid_ep10000.pth
qhybrid_ep2000.pth
qhybrid_ep3000.pth
qhybrid_ep4000.pth
qhybrid_ep5000.pth
qhybrid_ep6000.pth
qhybrid_ep7000.pth
qhybrid_ep8000.pth
qhybrid_ep9000.pth
quantum_hybrid_overnight_final.pth
quantum_hybrid_overnight_results.png
